**`validate_occupancy`**

Scores curated `occupancy_type` against hand-labeled ground-truth points
and gates on per-class F1, so a change that improves one class by
wrecking another fails loudly.

Aggregate agreement is deliberately not the gate. It sat near 65% while
Single-Family accuracy was 37.6%, so a single scalar looked adequate
while the largest error mode in the dataset went unnoticed. Recall alone
is not the gate either: it would fail a change that trades a little
recall for more precision, and pass one that inflates recall by
labelling everything a single class.

# Configure

In [ ]:
import argparse

import pandas as pd

from openplaces.io.curator.validation import score_classification

In [ ]:
parser = argparse.ArgumentParser(description='Validate curated occupancy.')
parser.add_argument('--recipe_id', default='US_footprint-cheer-2026')
parser.add_argument('--admin_ids', nargs='*', default=None)
# Default None resolves to cheer_linkage.BASELINE_PATH below: the baseline
# is survey-derived (third-party data), so it lives in the cache tree,
# never beside the recipe in the repository.
parser.add_argument('--baseline', default=None)
parser.add_argument('--write_baseline', action='store_true')
parser.add_argument('--tolerance', type=float, default=0.01)
parser.add_argument('--verbose', action='store_true')

# Test arguments

In [ ]:
ARGS_TEST = '--recipe_id US_footprint-cheer-2026 --verbose '

# Convert argument string to list of strings
args_list = [x for x in ARGS_TEST.split(' ') if x != '']

# Parse list of arguments
args = parser.parse_args(args_list)

# Display parsed arguments
args

# Validate occupancy against ground truth

In [ ]:
# CHEER-specific configuration (counties, survey path, band collapse)
# lives in scripts/cheer_linkage.py; the linkage and scoring themselves
# are generic and live in openplaces.io.curator.validation.
import sys

sys.path.insert(0, 'scripts')
from cheer_linkage import BASELINE_PATH, CLASSES, COUNTIES, link_ground_truth

counties = tuple(args.admin_ids) if args.admin_ids else COUNTIES
linked = link_ground_truth(counties, verbose=args.verbose)
linked.shape

In [ ]:
table = score_classification(
    linked['occupancy_type_canonical'],
    linked['predicted'],
    list(CLASSES),
)
table

In [ ]:
# Gate on F1 against the cache-tree baseline (or an explicit --baseline).
baseline_path = args.baseline or BASELINE_PATH
if args.write_baseline:
    table.to_csv(baseline_path, index=False)
    print(f'Baseline written: {baseline_path}')
else:
    baseline = pd.read_csv(baseline_path)
    merged = table.merge(baseline, on='class', suffixes=('', '_base'))
    merged['d_f1'] = merged['f1'] - merged['f1_base']
    print(merged[['class', 'f1_base', 'f1', 'd_f1']].to_string(index=False))
    regressed = merged[(merged['class'] != 'ALL') & (merged['d_f1'] < -args.tolerance)]
    if len(regressed):
        raise SystemExit(f'FAIL: {len(regressed)} class(es) lost F1')

---
# Convert to script

*The above line and heading identify the end of the script.*

In [ ]:
from openplaces.flow import convert_to_script

convert_to_script(commit=True)